In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import warnings
import random

warnings.filterwarnings('ignore')

In [2]:
df_placar2 = pd.read_csv("vlr_placar2.csv")
df_stats2 = pd.read_csv("vlr_stats2.csv")
df_merged = pd.read_csv("merged_status_placar.csv")
df_jog_map = pd.read_csv("jogador_por_mapas.csv")
df_matchups = pd.read_csv("matchups_score.csv")


In [3]:
df_placar2.head(3)

,URL,Mapa,Time1,Mapas_T1,Score_T1,Time2,Mapas_T2,Score_T2,Vencedor
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,Haven,LOUD,NaN,7,Cloud9,NaN,13,Cloud9
1,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,Abyss,LOUD,NaN,11,Cloud9,NaN,13,Cloud9
2,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,AllMaps,LOUD,0.0,18,Cloud9,2.0,26,Cloud9


In [4]:
df_stats2.head(3)

,URL,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Darker,LOUD,Viper,1.12,256,18,16,6,2,70,166,30,2,1,1
1,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,cauanzin,LOUD,Sova,0.98,209,15,13,3,2,75,152,26,0,1,-1
2,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Virtyy,LOUD,Yoru,0.70,178,14,16,4,-2,55,102,35,2,3,-1


In [5]:
df_merged.head(3)

,URL,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,...,HS%,FK,FD,+/-_FK_FD,Vencedor,Adversario,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Darker,LOUD,Viper,1.12,256,18,16,...,30,2,1,1,Cloud9,Cloud9,7.0,13.0,0.0,2.0
1,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,cauanzin,LOUD,Sova,0.98,209,15,13,...,26,0,1,-1,Cloud9,Cloud9,7.0,13.0,0.0,2.0
2,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Virtyy,LOUD,Yoru,0.70,178,14,16,...,35,2,3,-1,Cloud9,Cloud9,7.0,13.0,0.0,2.0


In [6]:
df_jog_map.head(3)

,Jogador,Time,Mapa,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Jogos,Wins,WinRate,Impacto
0,infiltrator,KRÜ,Abyss,1.360,293.5,43,30,6,13,87.5,186.0,32.0,5,2,3,2,0,0.0,1.383
1,lukxo,LOUD,Abyss,1.443,279.7,77,44,8,33,75.7,174.7,34.7,12,10,2,3,0,0.0,1.363
2,vora,100T,Abyss,1.385,255.5,33,22,6,11,82.0,163.5,28.5,2,2,0,2,1,0.5,1.304


In [7]:
df_matchups.head(3)

,Time,Adversario,Vitorias,Derrotas,Mapas,WinRate
0,100 Thieves,2Game Esports,2,0,2,100.0
1,100 Thieves,Cloud9,2,3,5,40.0
2,100 Thieves,Evil Geniuses,6,0,6,100.0


# Randon Florest 
Aplicação de Randon Florest para descobrir features importantes para a vitoria ou derrota

In [6]:
df = df_merged.copy()
df['Win'] = (df['Time'] == df['Vencedor']).astype(int)
cols_drop = [
    'Mapa',
    'Time',
    'Agente',
    'Vencedor',
    'Adversario',
    'Score_Time',
    'Score_Adv',
    'Mapas_Time',
    'Mapas_Adv',
    'URL',
    'Jogador'
]

df_model = df.drop(columns=cols_drop)

X = df_model.drop(columns=['Win'])
y = df_model['Win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

importances = model.feature_importances_

feat_importance = pd.Series(importances, index=X.columns)
feat_importance = feat_importance.sort_values(ascending=False)

print(feat_importance.head(20))

ACS          0.104594
R            0.103370
ADR          0.098341
HS%          0.096203
KAST         0.096041
+/-          0.082711
A            0.077237
D            0.076005
K            0.072748
+/-_FK_FD    0.057382
FK           0.051509
FD           0.049477
Mapa_Num     0.034383
dtype: float64


# Função de Criação de Times

In [6]:
COLS_SUM   = ["Score_Time", "Score_Adv", "Mapas_Time", "Mapas_Adv"]
COLS_MEAN  = ["R", "ACS", "K", "D", "A", "FK", "FD", "KAST", "ADR", "HS%"]
COLS_FATOR = ["R", "ACS", "K", "D", "A", "KAST", "ADR", "HS%", "FK", "FD"]

def vetor_jogador(
    jogador: str,
    df_merged: pd.DataFrame,
    df_jog_map: pd.DataFrame,
    mapa: str,
) -> dict | None:
    df_j = df_merged[df_merged["Jogador"] == jogador]
    if df_j.empty:
        return None

    stats = {}
    for c in COLS_SUM:
        if c in df_j.columns:
            stats[c] = df_j[c].sum()
    for c in COLS_MEAN:
        if c in df_j.columns:
            stats[c] = df_j[c].mean()

    stats["Jogador"] = jogador
    stats["Time"]    = df_j["Time"].iloc[0]
    stats["Mapa"]    = mapa

    df_jog = df_jog_map[df_jog_map["Jogador"] == jogador]
    if not df_jog.empty:
        stats["Jogos"]   = df_jog["Jogos"].sum()
        stats["Wins"]    = df_jog["Wins"].sum()
        stats["WinRate"] = round(
            (stats["Wins"] / stats["Jogos"] * 100) if stats["Jogos"] > 0 else 0.0, 1
        )
    else:
        stats["Jogos"]   = 0
        stats["Wins"]    = 0
        stats["WinRate"] = 0.0

    df_map = df_jog_map[
        (df_jog_map["Jogador"] == jogador) &
        (df_jog_map["Mapa"]    == mapa)
    ]
    if not df_map.empty:
        row_mapa    = df_map.iloc[0]
        media_geral = df_j[COLS_FATOR].mean()
        for c in COLS_FATOR:
            if c in stats and c in row_mapa.index and media_geral[c] != 0:
                stats[c] = stats[c] * (row_mapa[c] / media_geral[c])

    return stats


def sortear_time(
    df_merged: pd.DataFrame,
    df_jog_map: pd.DataFrame,
    mapa: str,
    n_jogadores: int = 5,
) -> pd.DataFrame:
    pool = df_merged["Jogador"].unique().tolist()
    vetores = []
    tentativas = 0

    while len(vetores) < n_jogadores and tentativas < len(pool) * 3:
        tentativas += 1
        jogador = random.choice(pool)
        if any(v["Jogador"] == jogador for v in vetores):
            continue
        v = vetor_jogador(jogador, df_merged, df_jog_map, mapa)
        if v:
            vetores.append(v)

    if len(vetores) < n_jogadores:
        print(f"[aviso] só foi possível montar {len(vetores)} jogadores para '{mapa}'")

    df_time = pd.DataFrame(vetores)

    cols_num = df_time.select_dtypes(include="number").columns
    media = df_time[cols_num].mean().round(2)

    for c in ["Jogos", "Wins"]:
        if c in df_time.columns:
            media[c] = df_time[c].sum()
    if "Jogos" in df_time.columns and media["Jogos"] > 0:
        media["WinRate"] = round(media["Wins"] / media["Jogos"] * 100, 1)

    media["Jogador"] = "MÉDIA TIME"
    media["Time"]    = "—"
    media["Mapa"]    = mapa

    df_time = pd.concat([df_time, media.to_frame().T], ignore_index=True)

    df_time["target"] = (
        pd.to_numeric(df_time["Score_Time"], errors="coerce") >
        pd.to_numeric(df_time["Score_Adv"],  errors="coerce")
    ).astype("Int64")

    cols_int  = ["K", "D", "A", "FK", "FD", "Score_Time", "Score_Adv",
                 "Mapas_Time", "Mapas_Adv", "Jogos", "Wins"]
    cols_dec  = ["R", "ACS", "KAST", "ADR", "HS%", "WinRate"]
    cols_info = ["Jogador", "Time", "Mapa"]

    for c in cols_int:
        if c in df_time.columns:
            df_time[c] = df_time[c].apply(lambda x: f"{float(x):.0f}" if pd.notna(x) else "—")
    for c in cols_dec:
        if c in df_time.columns:
            df_time[c] = df_time[c].apply(lambda x: f"{float(x):.2f}" if pd.notna(x) else "—")

    cols_ordem = cols_info + [c for c in cols_dec + cols_int if c in df_time.columns] + ["target"]
    return df_time[cols_ordem]


df_time = sortear_time(df_merged, df_jog_map, mapa="Haven")
df_time

,Jogador,Time,Mapa,R,ACS,KAST,ADR,HS%,WinRate,K,...,A,FK,FD,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv,Jogos,Wins,target
0,icy,EG,Haven,0.68,147.00,61.80,95.60,36.60,47.40,55,...,15,12,12,842,794,70,56,38,18,1
1,johnqt,SEN,Haven,1.21,184.60,76.60,122.40,24.80,56.50,104,...,75,5,9,1494,1428,145,125,69,39,1
2,skuba,NRG,Haven,1.16,189.30,77.30,134.80,28.00,60.50,88,...,20,7,8,966,808,101,69,43,26,1
3,s0m,NRG,Haven,1.02,189.70,73.90,127.70,30.60,49.00,99,...,71,10,16,1044,1008,91,91,49,24,1
4,Demon1,C9,Haven,0.83,207.00,63.50,137.00,34.00,40.00,29,...,4,8,6,546,544,44,40,25,10,1
5,MÉDIA TIME,—,Haven,0.98,183.52,70.62,123.50,30.80,52.20,75,...,37,8,10,978,916,90,76,224,117,1


In [10]:
df_time = sortear_time(df_merged, df_jog_map, mapa="Haven")
df_time

,Jogador,Time,Mapa,R,ACS,KAST,ADR,HS%,WinRate,K,...,A,FK,FD,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv,Jogos,Wins,target
0,skuba,NRG,Haven,1.16,189.30,77.30,134.80,28.00,60.50,88,...,20,7,8,966,808,101,69,43,26,1
1,N4RRATE,SEN,Haven,1.23,233.50,74.50,158.90,27.80,56.50,122,...,55,19,11,1494,1428,145,125,69,39,1
2,havoc,FUR,Haven,0.89,211.00,86.00,139.00,27.00,12.50,14,...,7,5,3,310,322,26,28,16,2,0
3,cauanzin,LOUD,Haven,0.80,168.80,65.70,116.40,27.70,31.10,113,...,35,13,17,892,1062,52,110,45,14,0
4,johnqt,SEN,Haven,1.21,184.60,76.60,122.40,24.80,56.50,104,...,75,5,9,1494,1428,145,125,69,39,1
5,MÉDIA TIME,—,Haven,1.06,197.44,76.02,134.30,27.06,49.60,88,...,38,10,10,1031,1010,94,91,242,120,1


In [11]:
cols = ["R", "ACS", "K", "D", "A", "KAST", "ADR", "HS%", "FK", "FD", "WinRate", "Jogos"]

for i in cols:
    df_time[i] = pd.to_numeric(df_time[i], errors='coerce').fillna(0).astype(int)
    
df_time.dtypes

Jogador       object
Time          object
Mapa          object
R              int64
ACS            int64
KAST           int64
ADR            int64
HS%            int64
WinRate        int64
K              int64
D              int64
A              int64
FK             int64
FD             int64
Score_Time    object
Score_Adv     object
Mapas_Time    object
Mapas_Adv     object
Jogos          int64
Wins          object
target         Int64
dtype: object

In [23]:
COLS_MODELO = ["R", "ACS", "ADR", "HS%", "WinRate", "K", "D", "A", "FK", "FD"]

def gerar_confronto(df_merged, df_jog_map, mapa: str) -> dict | None:
    t1 = sortear_time(df_merged, df_jog_map, mapa=mapa)
    t2 = sortear_time(df_merged, df_jog_map, mapa=mapa)

    media_t1 = t1[t1["Jogador"] == "MÉDIA TIME"].iloc[0]
    media_t2 = t2[t2["Jogador"] == "MÉDIA TIME"].iloc[0]

    confronto = {}
    for c in COLS_MODELO:
        if c in media_t1.index and c in media_t2.index:
            v1 = pd.to_numeric(media_t1[c], errors="coerce")
            v2 = pd.to_numeric(media_t2[c], errors="coerce")
            confronto[f"{c}_dif"] = v1 - v2  
            
    score_t1 = pd.to_numeric(media_t1.get("Score_Time", 0), errors="coerce")
    score_t2 = pd.to_numeric(media_t2.get("Score_Time", 0), errors="coerce")
    confronto["target"] = int(score_t1 > score_t2)

    return confronto


def gerar_dataset(df_merged, df_jog_map, mapa: str, n_confrontos: int = 500) -> pd.DataFrame:
    registros = []
    for _ in range(n_confrontos):
        c = gerar_confronto(df_merged, df_jog_map, mapa=mapa)
        if c:
            registros.append(c)
    return pd.DataFrame(registros)

mapas = ['Haven', 'Abyss', 'Corrode', 'Bind', 'Pearl', 'Breeze',
         'Split', 'Lotus', 'Icebox', 'Sunset', 'Ascent', 'Fracture']

dfs = []
for mapa in mapas:
    df_temp = gerar_dataset(df_merged, df_jog_map, mapa=mapa, n_confrontos=1000)
    df_temp["mapa"] = mapa  
    dfs.append(df_temp)

df_dataset = pd.concat(dfs, ignore_index=True)
print(f"\nDataset total: {df_dataset.shape}")
df_dataset.head()


Dataset total: (12000, 12)


,R_dif,ACS_dif,ADR_dif,HS%_dif,WinRate_dif,K_dif,D_dif,A_dif,FK_dif,FD_dif,target,mapa
0,0.18,15.48,5.70,10.16,-1.9,-28,-41,-16,-4,-3,0,Haven
1,-0.11,-32.05,-17.99,-2.43,12.6,-26,-16,-2,-7,-3,1,Haven
2,0.11,13.32,5.49,0.73,4.6,70,51,16,9,4,1,Haven
3,0.03,-5.55,1.46,-2.24,22.4,3,6,-3,-1,-3,1,Haven
4,-0.10,-3.32,-4.28,-2.95,-13.8,11,19,11,2,2,1,Haven


In [35]:
# Aplicação de ML



TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, brier_score_loss
import warnings
warnings.filterwarnings("ignore")

FEATS = ["R", "ACS", "K", "D", "A", "KAST", "ADR", "HS%", "FK", "FD"]

# ── target: WinRate ponderado por jogos ──────────────────────────────────────
def wr_ponderado(wins: float, jogos: float, k: float = 5.0) -> float:
    """
    Suaviza WinRate para jogadores com poucos jogos.
    k=5 significa que jogadores com menos de 5 jogos puxam para 0.5.
    """
    prior = 0.5
    return (wins + k * prior) / (jogos + k)


# ── vetor médio de um time num mapa ─────────────────────────────────────────
def vetor_time(jogadores: list, df_jog_map: pd.DataFrame, mapa: str) -> np.ndarray | None:
    rows = []
    for jog in jogadores:
        df_map = df_jog_map[(df_jog_map["Jogador"] == jog) & (df_jog_map["Mapa"] == mapa)]
        if df_map.empty:
            df_map = df_jog_map[df_jog_map["Jogador"] == jog]
        if df_map.empty:
            continue

        row   = df_map[FEATS].mean().values
        wins  = df_map["Wins"].sum()
        jogos = df_map["Jogos"].sum()
        wr    = wr_ponderado(wins, jogos)
        rows.append(np.append(row, wr))   # features + wr_ponderado

    if len(rows) < 5:
        return None
    return np.mean(rows, axis=0)


# ── gerar dataset sintético de confrontos ───────────────────────────────────
def gerar_confrontos(df_jog_map: pd.DataFrame, n: int = 6000) -> pd.DataFrame:
    jogadores = df_jog_map["Jogador"].unique().tolist()
    mapas     = df_jog_map["Mapa"].unique().tolist()
    registros = []

    while len(registros) < n:
        mapa   = random.choice(mapas)
        time_a = random.sample(jogadores, 5)
        time_b = random.sample(jogadores, 5)

        va = vetor_time(time_a, df_jog_map, mapa)
        vb = vetor_time(time_b, df_jog_map, mapa)
        if va is None or vb is None:
            continue

        diff  = va - vb
        label = 1 if va[-1] > vb[-1] else 0   # quem tem maior wr_ponderado vence

        registros.append((*diff, label))
        registros.append((*(-diff), 1 - label))

    feat_cols = [f"diff_{f}" for f in FEATS] + ["diff_WR"]
    return pd.DataFrame(registros, columns=feat_cols + ["label"])


# ── treinar e comparar os 3 modelos ─────────────────────────────────────────
def treinar_modelos(df_jog_map: pd.DataFrame, n_confrontos: int = 6000) -> dict:
    print("Gerando confrontos sintéticos...")
    df_train = gerar_confrontos(df_jog_map, n=n_confrontos)
    print(f"{len(df_train)} confrontos gerados. Balanceamento: {df_train['label'].mean():.2f}\n")

    X = df_train.drop(columns="label").values
    y = df_train["label"].values

    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X)

    candidatos = {
        "Logistic Regression": CalibratedClassifierCV(
            LogisticRegression(max_iter=1000), cv=5, method="isotonic"
        ),
        "Random Forest": CalibratedClassifierCV(
            RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
            cv=5, method="isotonic"
        ),
        "Gradient Boosting": CalibratedClassifierCV(
            GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                       learning_rate=0.05, random_state=42),
            cv=5, method="isotonic"
        ),
    }

    resultados = {}
    melhor_nome, melhor_score = None, 999

    for nome, modelo in candidatos.items():
        # cross-val com log-loss (quanto menor melhor)
        scores = cross_val_score(modelo, X_sc, y, cv=5,
                                 scoring="neg_log_loss", n_jobs=-1)
        log_loss_cv = -scores.mean()

        # treina no dataset completo para usar na predição
        modelo.fit(X_sc, y)

        resultados[nome] = {"modelo": modelo, "log_loss_cv": log_loss_cv}

        status = " ← melhor" if log_loss_cv < melhor_score else ""
        print(f"{nome:25s} | log-loss CV: {log_loss_cv:.4f}{status}")

        if log_loss_cv < melhor_score:
            melhor_score = log_loss_cv
            melhor_nome  = nome

    print(f"\nModelo escolhido: {melhor_nome}")
    return resultados, scaler, melhor_nome


# ── prever confronto entre 2 times ──────────────────────────────────────────
def prever_confronto(
    time_a: list,
    time_b: list,
    mapa: str,
    df_jog_map: pd.DataFrame,
    resultados: dict,
    scaler: StandardScaler,
    melhor_nome: str,
) -> None:
    va = vetor_time(time_a, df_jog_map, mapa)
    vb = vetor_time(time_b, df_jog_map, mapa)

    if va is None or vb is None:
        print("Jogador sem dados suficientes.")
        return

    diff        = (va - vb).reshape(1, -1)
    diff_scaled = scaler.transform(diff)

    print(f"\n{'─'*50}")
    print(f"Mapa   : {mapa}")
    print(f"Time A : {', '.join(time_a)}")
    print(f"Time B : {', '.join(time_b)}")
    print(f"{'─'*50}")
    print(f"{'Modelo':<25} | {'Prob A':>7} | {'Prob B':>7} | Favorito")
    print(f"{'─'*50}")

    for nome, res in resultados.items():
        prob_a = res["modelo"].predict_proba(diff_scaled)[0][1]
        prob_b = 1 - prob_a
        fav    = "Time A" if prob_a >= 0.5 else "Time B"
        mark   = " ★" if nome == melhor_nome else ""
        print(f"{nome:<25} | {prob_a*100:>6.1f}% | {prob_b*100:>6.1f}% | {fav}{mark}")


# ── pipeline ─────────────────────────────────────────────────────────────────
resultados, scaler, melhor_nome = treinar_modelos(df_jog_map, n_confrontos=6000)

# sorteia 2 times e prevê
pool  = df_jog_map["Jogador"].unique().tolist()
mapas = df_jog_map["Mapa"].unique().tolist()

mapa   = random.choice(mapas)
time_a = random.sample(pool, 5)
time_b = random.sample(pool, 5)

prever_confronto(time_a, time_b, mapa, df_jog_map, resultados, scaler, melhor_nome)